In [1]:
#import all needed libraries
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler,PolynomialFeatures, OneHotEncoder,  FunctionTransformer
from sklearn.pipeline import make_pipeline, Pipeline, clone
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.metrics import mean_absolute_error

class FeatureCreator(BaseEstimator, TransformerMixin):
    GENERATED_FEATURES = [
        "log_table",
        "x_table",
        "girdle_ratio",

    ]

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        else:
            self.n_features_in_ = X.shape[1]

        return self

    def transform(self, X):
        df = X.copy()

        df["log_table"] = np.log1p(df["table"])
        df["x_table"] = df["x"] / df["table"]
        df["girdle_ratio"] = df["x"]/df["y"]

        return df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            if not hasattr(self, "feature_names_in_"):
                raise ValueError(
                    "input_features must be provided when fitting on "
                    "data without column names."
                )
            input_features = self.feature_names_in_

        input_features = np.asarray(input_features, dtype=object)

        return np.concatenate([
            input_features,
            np.asarray(self.GENERATED_FEATURES, dtype=object),
        ])


df = pd.read_csv("XY_diamonds.csv")


df_train = df[df.isnull()['price']==False]

df_test = df[df.isnull()['price']==True]
df_test.drop(columns='price',inplace=True)

numeric_cols = df_train.select_dtypes(include="number").columns.drop("price")
categorical_cols = df_train.select_dtypes(include=["object", "string"]).columns

clean = True

if clean:
    #clean up outliers and unrealistic data
    lower = df_train[numeric_cols].quantile(0.001)
    upper = df_train[numeric_cols].quantile(0.999)

    mask = df_train[numeric_cols].ge(lower) & df_train[numeric_cols].le(upper)

    df_train = df_train[mask.all(axis=1)]

xyz_col = ["x","y","z"]

X = df_train.drop(columns='price')
Y = df_train['price']
#get a shuffled split for plotting
X_train, X_val, Y_train, Y_val = train_test_split(X,Y,test_size=0.1,random_state=1,shuffle=True)

NUM_NAN_impute_transform = ColumnTransformer(transformers=[
                                            ("num_imp",SimpleImputer(strategy='median'),xyz_col),
                                            ],remainder="passthrough",
                                            verbose_feature_names_out=False).set_output(transform="pandas")

CAT_NAN_impute_transform = ColumnTransformer(transformers=[
                                            ("cat_imp",SimpleImputer(strategy="most_frequent"),categorical_cols),
                                            ],remainder="passthrough",
                                            verbose_feature_names_out=False).set_output(transform="pandas")

ZERO_impute_transform = ColumnTransformer(transformers=[
                                            ("num_imp",SimpleImputer(missing_values=0,strategy='median'),xyz_col),
                                            ],remainder="passthrough",
                                            verbose_feature_names_out=False).set_output(transform="pandas")

cat_pipeline = Pipeline([("NAN_impute_Cat_cols",CAT_NAN_impute_transform),
                        ('encoder',OneHotEncoder(sparse_output=False, categories='auto', drop='first',handle_unknown="ignore"))
                        ])


numeric_clean_pipeline = Pipeline([
                                ("NAN_impute_num_cols", NUM_NAN_impute_transform),
                                ("ZERO_impute_num_cols", ZERO_impute_transform),
                                ("feature_addition",FeatureCreator()),
                                ])

base_preprocessor = ColumnTransformer(transformers=[
                                                    ("numeric", numeric_clean_pipeline, numeric_cols),
                                                    ("categorical", cat_pipeline, categorical_cols)
                                                    ],verbose_feature_names_out=False).set_output(transform="pandas")

full_preprocessor = Pipeline([
                                ("base_prep", base_preprocessor),
                                ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                                ("scaler", StandardScaler())
                            ])

class SegmentedLinearRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, preprocessor, segment_target=['carat'], n_bins=24, min_samples_per_bin=500, target_transformer=FunctionTransformer(np.log1p,inverse_func=np.expm1)):
        self.preprocessor = preprocessor
        self.segment_target = segment_target
        self.n_bins = n_bins
        self.min_samples_per_bin = min_samples_per_bin
        self.target_transformer = target_transformer

    def _get_group_keys(self, X_df, is_fit=False):
        keys = []
        for col in self.segment_target:
            if pd.api.types.is_numeric_dtype(X_df[col]):
                if is_fit:
                    # Calculate and store interior bin edges
                    vals = X_df[col].to_numpy()
                    edges = np.unique(np.quantile(vals, np.linspace(0, 1, self.n_bins + 1)))[1:-1]
                    self.bin_edges_[col] = edges
                
                # Digitize into integer bins (0 to len(edges))
                edges = self.bin_edges_[col]
                binned = pd.Series(np.digitize(X_df[col].to_numpy(), edges), index=X_df.index)
                keys.append(binned)
            else:
                # Categorical / string columns group by their raw values
                keys.append(X_df[col])
        return keys

    def fit(self, X, y):
        X_df = X.copy()
        # Preserve original indices for clean alignment
        y_arr = np.asarray(y).ravel()
        y_trans = pd.Series(self.target_transformer.transform(y_arr),index=X_df.index)
        # y_trans = pd.Series(np.log1p(np.asarray(y).ravel()), index=X_df.index)
        y_trans = y
        self.bin_edges_ = {}
        group_keys = self._get_group_keys(X_df, is_fit=True)

        # Global fallback model and bounds
        self.global_lo_ = float(y_trans.min())
        self.global_hi_ = float(y_trans.max())
        self.global_pipe_ = make_pipeline(clone(self.preprocessor), LinearRegression())
        self.global_pipe_.fit(X_df, y_trans)

        self.models_ = {}
        self.model_cap_ = {}

        for comb_tuple, group in X_df.groupby(group_keys, observed=True):
            n_samples = len(group)
            
            if n_samples >= self.min_samples_per_bin:
                y_seg = y_trans.loc[group.index]
                pipe = make_pipeline(clone(self.preprocessor), LinearRegression())
                pipe.fit(group, y_seg)
                
                self.models_[comb_tuple] = pipe
                self.model_cap_[comb_tuple] = (float(y_seg.min()), float(y_seg.max()))
            else:
                self.models_[comb_tuple] = self.global_pipe_
                self.model_cap_[comb_tuple] = (self.global_lo_, self.global_hi_)
                
        return self

    def predict(self, X):
        # Reset index locally to guarantee 0..N-1 positional indexing
        X_df = X.reset_index(drop=True)
        preds = np.empty(len(X_df), dtype=np.float64)

        group_keys = self._get_group_keys(X_df, is_fit=False)

        for comb_tuple, group in X_df.groupby(group_keys, observed=True):
            if len(group) == 0:
                continue
                        
            model = self.models_.get(comb_tuple, self.global_pipe_)
            preds[group.index] = model.predict(group)

            lo, hi = self.model_cap_.get(comb_tuple, (self.global_lo_, self.global_hi_))
            
            # pred_log = model.predict(group)
            # pred_log = pred_log*1.0028140703517587+(-0.019597989949748706)
            # bounded_pred = np.clip(pred_log, lo - 0.5, hi + 0.5)

            # Positional assignment safe from index mismatch
            # preds[group.index] = np.expm1(bounded_pred)
            
            # skewed_pred = pred_log*1.0028140703517587+(-0.019597989949748706)
            # preds[group.index] = self.target_transformer.inverse_transform(skewed_pred)
            
        return np.clip(preds, 10.0, None)


In [2]:
X_train_transformed = base_preprocessor.fit_transform(X_train)
X_val_transformed = base_preprocessor.transform(X_val)
full_preprocessor2 = Pipeline([ 
                                ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                                ("scaler2", StandardScaler())
                            ])

model2 = SegmentedLinearRegressor(
    preprocessor=full_preprocessor2, 
    segment_target=['x_table'], 
    n_bins=24, 
    min_samples_per_bin=500
)

# 10-fold CV evaluation
m_scores = [] # MAE
kf = KFold(n_splits=10, shuffle=True, random_state=1)
for train_index, test_index in kf.split(X_train, Y_train):
    model2.fit(base_preprocessor.fit_transform(X_train).iloc[train_index], Y_train.iloc[train_index])
    prediction = model2.predict(base_preprocessor.transform(X_train).iloc[test_index])
    m_scores.append([mean_absolute_error(Y_train.iloc[test_index], prediction)])

print(f'MAE: {np.mean(m_scores)}, std:{np.std(m_scores)}')

MAE: 520.6026325964193, std:8.526364444317707
